In [ ]:
"""
Creating configuration files for basins in Hawaii
"""
import os
import json
from pathlib import Path
import pandas as pd

# Paths
config_sac_dir = r"..\configurations\sac"
config_uhg_dir = r"..\configurations\unithg"
template_sac_json = os.path.join(config_sac_dir, "sac_params_16103000.json")
template_sac_yaml = os.path.join(config_sac_dir, "sac.control.16103000.yaml")
template_uhg_json = os.path.join(config_uhg_dir, "uhg_params_16103000.json")
template_uhg_yaml = os.path.join(config_uhg_dir, "uhg.control.16103000.yaml")
area_file = Path('C:/Users/hlohman/OneDrive - Research Triangle Institute/Documents/Python Scripts/pysac/usgs_point_attr_drainage_area.all.parquet')

# Read templates
with open(template_sac_json, 'r') as f:
    sac_json_content = json.load(f)

with open(template_sac_yaml, 'r') as f:
    sac_yaml_content = f.read()

with open(template_uhg_json, 'r') as f:
    uhg_json_content = json.load(f)

with open(template_uhg_yaml, 'r') as f:
    uhg_yaml_content = f.read()

# List of usgs_ids (dynamically extracted from pyMARRMot_input folder)
import glob
usgs_files = glob.glob(r"..\pyMARRMot_input\usgs-*_daily_marrmot.csv")
usgs_ids = sorted(set([os.path.basename(f).split('_')[0] for f in usgs_files]))  # usgs-16103000 format

# Get basin area dataframe
area_df = pd.read_parquet(area_file)

for usgs_id in usgs_ids:
    # Get USGS number
    usgs_num = usgs_id.split("-")[1]

    # Get basin area
    hru_area = area_df.loc[area_df['location_id'] == usgs_id, 'value'].item()  # float

    # Create sac json
    sac_json_copy = sac_json_content.copy()
    sac_json_copy["hru_id"] = [usgs_num]
    sac_json_copy["hru_area"] = [hru_area]
    with open(os.path.join(config_sac_dir, f"sac_params_{usgs_num}.json"), 'w') as f:
        json.dump(sac_json_copy, f, indent=4)
    
    # Create sac yaml
    sac_yaml_copy = sac_yaml_content.replace("16103000", usgs_num)
    with open(os.path.join(config_sac_dir, f"sac.control.{usgs_num}.yaml"), 'w') as f:
        f.write(sac_yaml_copy)
    
    # Create uhg json
    uhg_json_copy = uhg_json_content.copy()
    # Scale uh_ords by area ratio
    uh_ords_copy = uhg_json_copy["uh_ords"][0]
    hru_area_copy = uhg_json_copy["hru_area"][0]
    uh_ords_updated = [x * (hru_area / hru_area_copy) for x in uh_ords_copy]  # scale parameters based on area ratio
    # Update uhg_json_copy with new hru_id, hru_area, and scaled uh_ords
    uhg_json_copy["hru_id"] = [usgs_num]
    uhg_json_copy["hru_area"] = [hru_area]
    uhg_json_copy["uh_ords"] = [uh_ords_updated]
    with open(os.path.join(config_uhg_dir, f"uhg_params_{usgs_num}.json"), 'w') as f:
        json.dump(uhg_json_copy, f, indent=4)

    # Create uhg yaml
    uhg_yaml_copy = uhg_yaml_content.replace("16103000", usgs_num)
    with open(os.path.join(config_uhg_dir, f"uhg.control.{usgs_num}.yaml"), 'w') as f:
        f.write(uhg_yaml_copy)

print("Files created successfully.")

Files created successfully.
